<!-- ============================================================ -->
<!-- NOTEBOOK HEADER — MLOps Introductory Course on GCP           -->
<!-- ============================================================ -->

<div style="border-bottom: 3px solid #4285F4; padding-bottom: 12px; margin-bottom: 20px;">

<div style="display: flex; align-items: center; justify-content: space-between;">
  <div>
    <img src="https://www.isae-supaero.fr/wp-content/uploads/2025/03/logo.svg" width="180">
  </div>
  <div style="text-align: right;">
    <img src="https://user-images.githubusercontent.com/63151412/167391313-4683cc69-2bf6-4597-b767-5c18e2bbbfa0.png" width="180">
  </div>
</div>

# Lab 05a — RAG Corpus & Retrieval

**Course:** MLOps Introductory Course on GCP · M2 Data Science · ISAE-SUPAERO  
**Lab created by:** Headmind Partners AI & Blockchain  
**Estimated duration:** ~1h00

</div>

## 📋 Lab Overview

**Retrieval-Augmented Generation (RAG)** combines information retrieval with large language models to generate responses grounded in external knowledge. Instead of relying solely on training data, RAG systems dynamically fetch relevant documents to provide accurate, domain-specific answers — a critical pattern for enterprise AI.

In this first part, you will build the **retrieval backbone** of a RAG system using **Vertex AI RAG Engine**, Google Cloud's managed service for document indexing and semantic search.

### Learning Objectives

1. **Initialize** the Vertex AI SDK and configure a GCP project for RAG.
2. **Create a RAG corpus** with a specified embedding model.
3. **Import and chunk** documents from Cloud Storage into the corpus.
4. **Query** the retriever and **tune** parameters (`top_k`, `distance_threshold`).
5. **Analyze** how chunking and retrieval settings affect search quality.

### Notebook Structure

| # | Section | Focus |
|---|---------|-------|
| 0 | Setup | Install dependencies, authenticate, configure project |
| 1 | RAG Corpus Creation | Create a corpus with embedding model configuration |
| 2 | Document Import | Import documents with chunking parameters |
| 3 | Retrieval Testing | Query the corpus, tune retrieval parameters |

### How to Read This Notebook

- **`# TODO`** — Code you need to write. Look for the `######` delimiters.
- **`✏️ Question`** — A conceptual question. Write your answer in the markdown cell below it.
- Cells **without** a TODO are provided — read them, run them, and make sure you understand them.
- Documentation links are provided in 📖 callouts whenever a new API is introduced.

---
## 0 · Setup

### 0.1 Install dependencies

We need the Vertex AI SDK with RAG Engine support.

In [ ]:
%pip install --upgrade google-cloud-aiplatform -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 56.2 MB/s eta 0:00:00


### 0.2 Authentication

If you are running this notebook **locally** (not on Vertex AI Workbench), run the next cell to authenticate with your GCP credentials:

> 📖 **Docs:** [Install the gcloud CLI](https://cloud.google.com/sdk/docs/install)

In [ ]:
#!gcloud auth application-default login

### 0.3 Imports

In [ ]:
# ── Standard library ──
import time
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Vertex AI ──
import vertexai
from vertexai import rag
from vertexai.generative_models import GenerativeModel, Tool

print(f"✅ Imports complete")
print(f"Vertex AI SDK version: {vertexai.__version__}")

✅ Imports complete
Vertex AI SDK version: 1.141.0


### 0.4 Configuration

> 📖 **Vertex AI RAG Engine documentation:**
> - [RAG Engine overview](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-overview)
> - [RAG quickstart](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-quickstart)
> - [Supported regions](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/locations)

> ⚠️ **Important:** Verify that your region supports RAG Engine. `europe-west3` is supported.

In [34]:
##############################  TODO  ##############################
# Configure your GCP project and RAG corpus settings.
# - PROJECT_ID: find it in the GCP Console project selector.
# - YOUR_NAME: your trigram (e.g. SUC for Sasuke Uchiha).

PROJECT_ID = "isae-sdd-481407"  # @param {type:"string"}
LOCATION = "europe-west3"  # @param {type:"string"}
BUCKET_NAME = "bucket-lab05"  # @param {type:"string"}
YOUR_NAME = "mregaieg"  # @param {type:"string"}
EXPERIMENT_ID = "exp3"  # @param {type:"string"}
# Change this for each experiment: "baseline", "exp1", "exp2"
# On first notebook run, keep it as "baseline"
####################################################################

DISPLAY_NAME = f"rag-corpus-{YOUR_NAME}-{EXPERIMENT_ID}"
PATHS = [f"gs://{BUCKET_NAME}/{YOUR_NAME}/documents/"]
####################################################################

# Initialize Vertex AI SDK
vertexai.init(project=PROJECT_ID, location=LOCATION)

print(f"✅ Vertex AI initialized")
print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Corpus name: {DISPLAY_NAME}")

✅ Vertex AI initialized
Project: isae-sdd-481407
Location: europe-west3
Corpus name: rag-corpus-mregaieg-exp3


---
## 1 · RAG Corpus Creation

A **RAG corpus** is a collection of documents indexed with vector embeddings for semantic search. Vertex AI RAG Engine manages the full pipeline: chunking, embedding, indexing, and serving.

By default, RAG Engine uses **RagManagedDb** as the vector database — no manual setup required.

We'll use Google's **text-embedding-005** model (768 dimensions), the latest version optimized for semantic similarity.

> 📖 **Docs:**
> - [RAG quickstart](https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-quickstart)
> - [Text embedding models](https://cloud.google.com/vertex-ai/generative-ai/docs/embeddings/get-text-embeddings)


In [35]:
##############################  TODO  ##############################
# 1. Create an embedding model config using RagEmbeddingModelConfig
#    with a VertexPredictionEndpoint pointing to text-embedding-005.
#    Model path format: "publishers/google/models/<model-name>"
#
# 2. Create the RAG corpus using rag.create_corpus() with:
#    - display_name: your DISPLAY_NAME
#    - backend_config: a RagVectorDbConfig wrapping your embedding config
#
# 📖 Hint: see the quickstart at
#    https://cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/rag-quickstart

embedding_model_config = rag.RagEmbeddingModelConfig(
    vertex_prediction_endpoint=rag.VertexPredictionEndpoint(
        publisher_model="publishers/google/models/text-embedding-004"  # TODO
    )
)

rag_corpus = rag.create_corpus(
    display_name=DISPLAY_NAME,  # TODO
    backend_config=rag.RagVectorDbConfig(
        rag_embedding_model_config=embedding_model_config  # TODO
    ),
)
####################################################################

print(f"✅ RAG corpus created")
print(f"Corpus resource name: {rag_corpus.name}")
print(f"Embedding model: text-embedding-005 (768 dimensions)")

✅ RAG corpus created
Corpus resource name: projects/588027604890/locations/europe-west3/ragCorpora/4604930618986332160
Embedding model: text-embedding-005 (768 dimensions)


**✏️ Question 1 — Embedding Models**

a) Name **two other embedding models** (from any provider) that could be used in a RAG system.  
b) What factors would you consider when choosing an embedding model for a **multilingual** RAG application?

---
*Your answer:*

 a) Two other embedding models that could be used in a RAG system are **OpenAI’s text-embedding-3-large** and **Cohere’s embed-multilingual-v3.0**. .

 b) For a multilingual RAG application, I would consider factors such as **language support**, **cross-lingual retrieval ability** (matching queries and documents in different languages), **embedding dimension size**, **latency and cost**, and the **maximum input length** the model can handle. These factors affect retrieval accuracy, system performance, and storage requirements.


---

---
## 2 · Document Import

### 2.1 Prerequisites

Before importing, make sure you have run the **`pdf_to_bucket.ipynb`** utility notebook to convert your PDFs to `.txt` and upload them to Cloud Storage.

> 💡 **Tip:** Vertex AI RAG Engine now supports direct PDF import, but for this lab we use `.txt` files to understand the full preprocessing pipeline.

### 2.2 Import files into corpus

The key chunking parameters are:

- **`chunk_size`**: Maximum tokens per chunk (typical: 512–1024). Smaller = more precise embedding but less context per chunk.
- **`chunk_overlap`**
- **`max_embedding_requests_per_min`**: Rate limit to avoid quota errors.

> 📖 **Docs:**
> - [`rag.import_files()`](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/samples/generativeaionvertexai-rag-import-files?hl=en)
> - [Chunking strategies](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/rag-engine/fine-tune-rag-transformations)

**✏️ Question 2 — Chunking Strategy**

a) What is **chunk overlap** and why is it necessary in RAG systems?  
b) What problems might arise if you set `chunk_overlap=0`?  
c) A colleague suggests using `chunk_size=2048` with `chunk_overlap=0`. In what scenario could this be a reasonable choice?

---
*Your answer:*

 a) **Chunk overlap** means repeating a small part of one chunk at the start of the next chunk. It is necessary because it helps preserve context between chunks, so important information is not lost when the text is split.

 b) If `chunk_overlap = 0`, some ideas or sentences may be split between two chunks. This can cause **loss of context**, **lower retrieval accuracy**, and incomplete results, because the retriever may fail to find the full meaning of the text.

 c) Using `chunk_size = 2048` with `chunk_overlap = 0` can be reasonable when the text is already made of **independent, self-contained sections**, such as FAQ entries, short product descriptions, or other structured records. In this case, each chunk can stand on its own, so overlap is less important.

---

In [37]:
##############################  TODO  ##############################
# Import your documents into the corpus with chunking.
# Recommended starting values:
#   chunk_size: 512   |   chunk_overlap: 100   |   rate limit: 200

response = rag.import_files(
    corpus_name= rag_corpus.name,
    paths=PATHS,
    transformation_config=rag.TransformationConfig(
        rag.ChunkingConfig(chunk_size=1024, chunk_overlap=100),
    ),
    import_result_sink="gs://bucket-lab05/mregaieg/documents/",
    max_embedding_requests_per_min=200,
)
####################################################################

print(f"✅ Documents imported to corpus")
print(f"Response: {response}")

✅ Documents imported to corpus
Response: skipped_rag_files_count: 2



> 💡 **Tip:** The import is asynchronous. For large document sets, check progress in the **Vertex AI Console → RAG Engine**.

---
## 3 · Retrieval Testing

Before connecting retrieval to a language model, it's important to test the retriever **in isolation**.

Key parameters:
- **`top_k`**: Number of most similar chunks to return (typical: 3–10).
- **`vector_distance_threshold`**: Only return chunks closer than this threshold (0–1, lower = more similar).

> 📖 **Docs:**
> - [`RagRetrievalConfig`](https://docs.cloud.google.com/php/docs/reference/cloud-ai-platform/latest/V1.RagRetrievalConfig)
> - [`rag.retrieval_query()`](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/samples/generativeaionvertexai-rag-retrieval-query?hl=en)
> - [More documentation :)](https://deepwiki.com/googleapis/python-aiplatform/10-development-and-testing#retrieval-and-search-configuration)


In [43]:
##############################  TODO  ##############################
# Configure and test retrieval. Experiment with:
#   top_k: 3–10    |    vector_distance_threshold: 0.3–0.7
#
# Try different queries: specific facts, broad topics, out-of-context.

rag_retrieval_config = rag.RagRetrievalConfig(
    top_k=5,  # TODO
    filter=rag.Filter(vector_distance_threshold=0.6),  # TODO
)

response = rag.retrieval_query(
    rag_resources=[
        rag.RagResource(rag_corpus=rag_corpus.name)  # TODO
    ],
    text="Who is Cosette?",  # TODO — try other queries!
    rag_retrieval_config=rag_retrieval_config,
)
####################################################################

print(f"✅ Retrieved {len(response.contexts.contexts)} chunks\n")
for i, ctx in enumerate(response.contexts.contexts, 1):
    print(f"─── Chunk {i} (cosine distance: {ctx.score:.3f}) ───")
    print(ctx.text[:200] + "...\n")

✅ Retrieved 5 chunks

─── Chunk 1 (cosine distance: 0.302) ───
Il n'est pas rare aujourd'hui que le garçon bouvier se nomme Arthur, Alfred ou Alphonse, et que le vicomte – s'il y a encore des vicomtes – se nomme Thomas, Pierre ou Jacques. Ce déplacement qui met l...

─── Chunk 2 (cosine distance: 0.332) ───
Doux être faible qui ne devait rien comprendre à ce monde ni à Dieu, sans cesse punie, grondée, rudoyée, battue et voyant à côté d'elle deux petites créatures comme elle, qui vivaient dans un rayon d'...

─── Chunk 3 (cosine distance: 0.348) ───
Elle reçut la lettre, et la froissa dans ses mains tout le jour. Le soir elle entra chez un barbier qui habitait le coin de la rue, et défit son peigne. Ses admirables cheveux blonds lui tombèrent jus...

─── Chunk 4 (cosine distance: 0.352) ───
Cette rougeur dura peu. La soeur leva sur Fantine son oeil calme et triste, et dit :

– Monsieur le maire est parti.

Fantine se redressa et s'assit sur ses talons. Ses yeux étincelèrent. Une joie...

> 💡 **Experimentation tips:**
> - Try queries with different specificity levels (broad vs. narrow)
> - Ask an **out-of-context** question (e.g., "What is the capital of France?") and observe the distances
> - Compare results with different `top_k` and `distance_threshold` values

**✏️ Question 3 — Retrieval Tuning**

a) You query "What is reinforcement learning?" against a corpus about the DeepSeek-R1 paper. The top 5 chunks all have distance > 0.7. What does this tell you, and how should you adjust the threshold?  
b) What is the role of the **distance threshold** in production RAG systems? What happens if it is too low? Too high?  
c) Why is it useful to test the retriever **separately** before connecting it to the LLM?

---
*Your answer:*

 a)If the top 5 chunks all have a distance greater than 0.7, it means the retrieved chunks are not very similar to the query. This suggests that the corpus may not contain relevant information about reinforcement learning, or that the threshold is too strict. To allow more results to be returned, the **distance threshold should be increased** slightly.

 b) The **distance threshold** controls how similar a document chunk must be to the query in order to be retrieved. In production RAG systems, it helps filter out irrelevant chunks and keep only useful context. If the threshold is **too low**, the retriever may be too strict and sometimes return **0 chunks**, which means the LLM gets no context. If the threshold is **too high**, more chunks will be returned, but some of them may be irrelevant, which can reduce answer quality.

 c) Testing the retriever separately helps verify that the system is retrieving relevant chunks before sending them to the LLM. This makes it easier to debug retrieval problems and ensure that the model receives good context, which improves the quality of the final answers.

---

### 3.1 Save corpus reference for Lab 05b

Copy the corpus resource name below — you will need it in the next notebook.

In [44]:
print(f"\n📋 Copy this corpus resource name for Lab 05b:")
print(f"   {rag_corpus.name}")


📋 Copy this corpus resource name for Lab 05b:
   projects/588027604890/locations/europe-west3/ragCorpora/4604930618986332160


---
## Summary

In this lab, you learned to:

| Step | What you did | Tool / Feature used |
|------|-------------|---------------------|
| **Setup** | Configured GCP project and initialized Vertex AI | `vertexai.init()` |
| **Corpus creation** | Created a RAG corpus with text-embedding-005 | `rag.create_corpus()`, `RagEmbeddingModelConfig` |
| **Document import** | Imported chunked documents from Cloud Storage | `rag.import_files()`, `ChunkingConfig` |
| **Retrieval testing** | Queried the retriever and tuned parameters | `rag.retrieval_query()`, `RagRetrievalConfig` |

**Next lab:** In **Lab 05b**, you will connect this retriever to a Gemini model for grounded generation and evaluate the full RAG system using RAGAS metrics.